# Experiment Runner with Automatic Data Loading

This notebook demonstrates the ExperimentRunner which:
1. **Loads experiment configuration from YAML** - including data loading config
2. **Automatically loads and parses data** - using configured loader and parser
3. **Runs multiple RAG configs** - either sequentially or in parallel
4. **Generates detailed reports** - per-query and aggregate metrics
5. **Compares results** - across different configurations

The experiment configuration in YAML specifies:
- Data source (HuggingFace dataset)
- Data parser (e.g., title_passage)
- RAG config directory
- Number of queries to evaluate
- Parallel execution settings

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [2]:
import os
from dotenv import load_dotenv

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner

load_dotenv(override=True)
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

/Users/adityanarayan/git/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
# Load experiment configuration from YAML
EXPERIMENT_CONFIG_PATH = "experiment_configs/covidqa_experiment.yaml"
experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Cache Dir:   {experiment_config.cache_dir}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  config
  Report Dir:  reports
  Cache Dir:   cache
  Num Queries: 246
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'covidqa', 'split': 'test', 'limit': 246}

Data Parser:
  Type: title_passage


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.sections[0].per_query)
    
    # Display aggregate statistics
    print("\nAggregate Statistics:")
    display(report.sections[0].summary)

In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/covidqa (test)...
Loaded 246 samples

✅ Data loaded successfully!
  Documents: 984 parsed documents
  Raw Data:  246 samples

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f"Loaded {len(configs)} RAG pipeline configurations:")
for cfg in configs:
    print(f"  - {cfg.name}")
    print(f"    Chunking: {cfg.chunking.type.value}")
    print(f"    Retrieval Pipeline: ")
    for search in cfg.retrieval.search.searches:
        print(f"      - {search.type.value}")
    print(f"    Reranker: {cfg.retrieval.rerank.type.value}")
    print(f"    Generation: {cfg.generation.config['model']}")

Loaded 3 RAG pipeline configurations:
  - covidqa_biomedical_multiquery_v1
    Chunking: token
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: qwen/qwen3-32b
  - covidqa_multiquery_hybrid
    Chunking: semantic
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.1-8b-instant
  - covidqa_multiquery_hybrid_v2
    Chunking: semantic
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile


## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [7]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 3 configurations from 0 to 246 queries...
Parallel mode: False
Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57983.08it/s]


Loading reranker model: BAAI/bge-reranker-v2-gemma


Loading weights: 100%|██████████| 164/164 [00:00<00:00, 4125.38it/s]


Progress: 246/246 (100.0%) | QPS: 231.31 | ETA: 0s | Elapsed: 1s0s
Loading embedding model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6896.79it/s]


Loading reranker model: BAAI/bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1708.30it/s]


Progress: 246/246 (100.0%) | QPS: 241.88 | ETA: 0s | Elapsed: 1s 0s
Progress: 246/246 (100.0%) | QPS: 244.55 | ETA: 0s | Elapsed: 1s: 0s

✅ Experiments completed!
  Ran 3 configurations
  - covidqa_biomedical_multiquery_v1: 246 queries
  - covidqa_multiquery_hybrid: 246 queries
  - covidqa_multiquery_hybrid_v2: 246 queries


## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [8]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Found 3 configs with JSONL files
Using key #0: ****fyaJ
Using key #0: ****fyaJ


Record 30 evaluation failed: Expecting ',' delimiter: line 237 column 6 (char 6635)


Using key #0: ****fyaJ
Using key #0: ****fyaJ
Using key #0: ****fyaJ
Using key #0: ****fyaJ
429 on ****fyaJ
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kw7e089nf7yba8zdzhf2rn33` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11901, Requested 3337. Please try again in 16.189999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sat, 11 Jul 2026 04:53:00 GMT
  content-type: application/json
  content-length: 391
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 17
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 995
  x-ratelimit-remaining-tokens: 99
  x-ratelimit-reset-requests: 7m12s
  x-ratelim

Record 183 evaluation failed: Expecting ',' delimiter: line 232 column 6 (char 7266)


Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
429 on ****Z8Yy
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kq0qh6m2e87rj92vbs1qjw6s` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11445, Requested 3037. Please try again in 12.41s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sat, 11 Jul 2026 04:57:43 GMT
  content-type: application/json
  content-length: 384
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 13
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 992
  x-ratelimit-remaining-tokens: 555
  x-ratelimit-reset-requests: 11m31.2s
  x-ratelimit-reset-tokens: 57.225s
 

Record 189 evaluation failed: Expecting ',' delimiter: line 144 column 6 (char 7532)


Using key #0: ****fyaJ
Using key #0: ****fyaJ
Using key #0: ****fyaJ
429 on ****fyaJ
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kw7e089nf7yba8zdzhf2rn33` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10744, Requested 3509. Please try again in 11.265s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sat, 11 Jul 2026 04:58:12 GMT
  content-type: application/json
  content-length: 385
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 12
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 986
  x-ratelimit-remaining-tokens: 1256
  x-ratelimit-reset-requests: 20m9.6s
  x-ratelimit-reset-tokens: 53.72s
 

Record 14 evaluation failed: All Groq API keys are currently rate limited.
Record 21 evaluation failed: All Groq API keys are currently rate limited.
Record 22 evaluation failed: All Groq API keys are currently rate limited.
Record 23 evaluation failed: All Groq API keys are currently rate limited.
Record 24 evaluation failed: All Groq API keys are currently rate limited.
Record 25 evaluation failed: All Groq API keys are currently rate limited.
Record 26 evaluation failed: All Groq API keys are currently rate limited.
Record 27 evaluation failed: All Groq API keys are currently rate limited.
Record 28 evaluation failed: All Groq API keys are currently rate limited.
Record 29 evaluation failed: All Groq API keys are currently rate limited.
Record 30 evaluation failed: All Groq API keys are currently rate limited.
Record 31 evaluation failed: All Groq API keys are currently rate limited.
Record 32 evaluation failed: All Groq API keys are currently rate limited.
Record 33 evaluation fail

429 on ****Z8Yy
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kq0qh6m2e87rj92vbs1qjw6s` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10134, Requested 2297. Please try again in 2.155s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sat, 11 Jul 2026 05:06:26 GMT
  content-type: application/json
  content-length: 384
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 3
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 955
  x-ratelimit-remaining-tokens: 1866
  x-ratelimit-reset-requests: 1h4m48s
  x-ratelimit-reset-tokens: 50.67s
  x-request-id: req_01kx7s3p83fe6awyara0mgn3qr
  via: 1.1 google
  cf-ca

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [9]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [10]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: covidqa_biomedical_multiquery_v1

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `covidqa_biomedical_multiquery_v1`

**name**: covidqa_biomedical_multiquery_v1  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'token', 'config': {'max_tokens': 120, 'overlap_tokens': 20}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'pritamdeka/S-PubMedBert-MS-MARCO', 'dimension': 768}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 768}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 80}}, {'type': 'sparse', 'config': {'top_k': 80}}]}, 'query_transform': {'type': 'multi_query', 'provider': 'groq', 'config': {'model': 'qwen/qwen3-32b', 'num_queries': 3, 'temperature': 0.0, 'max_tokens': 192}}, 'fusion': {'type': 'rrf', 'config': {'k': 60}}, 'rerank': {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-gemma', 'top_k': 15}}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'qwen/qwen3-32b', 'temperature': 0.0, 'max_tokens': 512, 'system_prompt': 'You are an expert biomedical question answering assistant.\n\nAnswer ONLY using the retrieved context.\n\nDo not use prior knowledge.\n\nIf the answer cannot be determined from the retrieved context,\nreply exactly:\n\n"Insufficient evidence in retrieved context."\n\nKeep answers concise, factual and evidence-based.\n', 'user_prompt': 'Retrieved Context\n-----------------\n{context}\n-----------------\n\nQuestion:\n{query}\n\nAnswer using only the retrieved context.'}}  •  **evaluation**: None  •  **cache**: {'enabled': True, 'cache_dir': 'cache'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,Which viruses may not cause prolonged inflammation due to strong induction o...,"[{'text': '', 'title': '', 'content': 'are also activated directly by patter...",Insufficient evidence in retrieved context.,0.6667,0.4118,0.2549,0.0,0.1765,-0.1765,0.0,0.4286,-0.4286,0.0,0.0,0.0
1,When was the first case of COVID-19 identified?,"[{'text': '', 'title': '', 'content': 'Any declarative claim that is inconsi...",Insufficient evidence in retrieved context.,0.2857,0.2692,0.0165,0.0,0.0769,-0.0769,0.0,0.2857,-0.2857,1.0,1.0,0.0
2,How many antigens could be detected by Liew's multiplex ELISA test?,"[{'text': '', 'title': '', 'content': 'LAIV is administered as a nasal spray...",Insufficient evidence in retrieved context.,0.0000,0.1250,-0.1250,0.0,0.0625,-0.0625,0.0,0.5000,-0.5000,1.0,1.0,0.0
3,What is the structure of Hantaan virus?,"[{'text': '', 'title': '', 'content': 'In recent years, the infection rate o...",Insufficient evidence in retrieved context.,0.3000,0.3000,0.0000,0.0,0.3000,-0.3000,0.0,1.0000,-1.0000,1.0,1.0,0.0
4,How many people did SARS-CoV infect?,"[{'text': '', 'title': '', 'content': 'We detail the first European cases. A...",Insufficient evidence in retrieved context.,0.0556,0.0667,-0.0111,0.0,0.0667,-0.0667,0.0,1.0000,-1.0000,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,What is a conclusion of this report?,"[{'text': '', 'title': '', 'content': 'results.', 'metadata': {'doc_id': 'sa...",Insufficient evidence in retrieved context.,NaN,0.1250,NaN,NaN,0.1250,NaN,NaN,1.0000,NaN,NaN,1.0,NaN
242,What are the results of filamentous phage immunizations in mice?,"[{'text': '', 'title': '', 'content': 'intraperitoneal infections, between 2...",Insufficient evidence in retrieved context.,NaN,0.2222,NaN,NaN,0.2222,NaN,NaN,0.7500,NaN,NaN,1.0,NaN
243,What is marked in green?,"[{'text': '', 'title': '', 'content': 'the trachea, lung, brain, heart, live...",Insufficient evidence in retrieved context.,NaN,0.1053,NaN,NaN,0.1053,NaN,NaN,1.0000,NaN,NaN,1.0,NaN
244,How can CCR5's effect in HIV-1 transmission be reduced?,"[{'text': '', 'title': '', 'content': 'virus . Like NDV, PIV5 has a 3'-to 5'...",Insufficient evidence in retrieved context.,NaN,0.2105,NaN,NaN,0.1053,NaN,NaN,0.5000,NaN,NaN,0.0,NaN


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.2874,0.2887,0.2741,0.1684,0.2477
1,utilization_score,0.0000,0.1811,0.0000,0.1288,0.1269
2,completeness_score,0.0000,0.6631,0.0000,0.2940,0.6283
3,adherence_score,0.6842,0.8415,0.4648,0.3652,0.2632


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [11]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: reports/comparison.csv


In [12]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,covidqa_high_recall_hybrid_v2,relevance_score,0.2687,0.2887,0.2600,0.1684,0.2288
1,covidqa_hyde_hybrid,relevance_score,0.2273,0.2887,0.2582,0.1684,0.2374
2,covidqa_multiquery_hybrid,relevance_score,0.3594,0.2887,0.3022,0.1684,0.2523
3,covidqa_multiquery_hybrid_v2,relevance_score,0.2874,0.2887,0.2741,0.1684,0.2477
4,covidqa_biomedical_multiquery_v1,relevance_score,0.1197,0.2887,0.2227,0.1684,0.2639


In [ ]:
from reporting.base import Report


report = Report.from_jsonl(
    "temp/covidqa_high_recall_hybrid_v2.jsonl",
    config_name="covidqa_high_recall_hybrid_v2",
)

report.display()

report.save_json("reports/covidqa_high_recall_hybrid_v2.json")

AttributeError: type object 'Report' has no attribute 'from_jsonl'

Progress: 102/246 (41.5%) | QPS: 0.13 | ETA: 19.2m | Elapsed: 13.6m

## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons